# Stock Market Exploratory Data Analysis
### Full Universe — All Alpaca-Tradeable US Equities (12,500+ symbols)

This notebook loads, explores, and analyzes the complete set of daily OHLCV stock data stored as Parquet files on the Z: drive.

**Data source:** `Z:\market_data\1Day\` — one Parquet file per symbol  
**Schema:** DatetimeIndex(`timestamp`), columns: `open`, `high`, `low`, `close`, `volume`

## 1. Import Required Libraries

In [ ]:
import sys
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Add project root to path so we can import src modules
PROJECT_ROOT = Path(r"c:\Users\hdroe\OneDrive\Documents\Coursera\alpaca-trading-api")
sys.path.insert(0, str(PROJECT_ROOT))

from src.trading.data import DataStore

# Plot settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100
warnings.filterwarnings("ignore", category=FutureWarning)

print("Libraries loaded ✓")

## 2. Configure Data Store & Discover All Available Symbols

In [ ]:
DATA_DIR = Path(r"Z:\market_data")
PARQUET_DIR = DATA_DIR / "1Day"

# Initialize DataStore (same class used by the download scripts)
store = DataStore(data_dir=str(DATA_DIR), organize_by_timeframe=True)

# Discover all symbols by scanning parquet files
parquet_files = sorted(PARQUET_DIR.glob("*.parquet"))
all_symbols = [f.stem for f in parquet_files]

print(f"Data directory:    {PARQUET_DIR}")
print(f"Total symbols:     {len(all_symbols):,}")
print(f"First 10 symbols:  {all_symbols[:10]}")
print(f"Last 10 symbols:   {all_symbols[-10:]}")

## 3. Load Inventory & Display Dataset Statistics

In [ ]:
# Build inventory by scanning file sizes and reading metadata from a sample
file_stats = []
for f in parquet_files:
    file_stats.append({
        "symbol": f.stem,
        "file_size_mb": f.stat().st_size / (1024 * 1024),
    })

inventory_df = pd.DataFrame(file_stats)

print(f"Total symbols:          {len(inventory_df):,}")
print(f"Total storage:          {inventory_df['file_size_mb'].sum():.2f} MB  ({inventory_df['file_size_mb'].sum()/1024:.2f} GB)")
print(f"Avg file size:          {inventory_df['file_size_mb'].mean():.3f} MB")
print(f"Median file size:       {inventory_df['file_size_mb'].median():.3f} MB")
print()

# Distribution of file sizes
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(inventory_df["file_size_mb"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("File Size (MB)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Parquet File Sizes")

axes[1].hist(inventory_df["file_size_mb"].clip(upper=0.5), bins=50, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_xlabel("File Size (MB) — clipped at 0.5 MB")
axes[1].set_ylabel("Count")
axes[1].set_title("Zoomed: File Size Distribution")
plt.tight_layout()
plt.show()

## 4. Sample & Load Multiple Stocks into a Combined DataFrame

We systematically sample ~25 symbols across the alphabet so the analysis spans the full breadth of the universe — not just the usual suspects.

In [ ]:
# Systematic sample: pick every Nth symbol to spread across the alphabet
SAMPLE_SIZE = 25
step = max(1, len(all_symbols) // SAMPLE_SIZE)
sample_symbols = all_symbols[::step][:SAMPLE_SIZE]
print(f"Sampled {len(sample_symbols)} symbols: {sample_symbols}")

# Load each symbol and combine
frames = []
for sym in sample_symbols:
    try:
        df = store.load(sym, "1Day")
        if df is not None and len(df) > 0:
            df = df.copy()
            df["symbol"] = sym
            frames.append(df)
    except Exception as e:
        print(f"  Skipping {sym}: {e}")

combined_df = pd.concat(frames)
print(f"\nCombined shape: {combined_df.shape}")
print(f"Symbols loaded: {combined_df['symbol'].nunique()}")
print(f"Date range:     {combined_df.index.min()} → {combined_df.index.max()}")
combined_df.head(10)

## 5. Compute Summary Statistics Across All Symbols

We iterate over a large random sample (500+ symbols) and compute per-symbol stats: mean close, total volume, trading days, and date range.

In [ ]:
# Sample 500 random symbols for broad summary stats
np.random.seed(42)
stat_sample = np.random.choice(all_symbols, size=min(500, len(all_symbols)), replace=False)

summary_rows = []
for sym in stat_sample:
    try:
        df = pd.read_parquet(PARQUET_DIR / f"{sym}.parquet")
        if len(df) == 0:
            continue
        summary_rows.append({
            "symbol": sym,
            "mean_close": df["close"].mean(),
            "median_close": df["close"].median(),
            "total_volume": df["volume"].sum(),
            "avg_daily_volume": df["volume"].mean(),
            "trading_days": len(df),
            "start_date": df.index.min(),
            "end_date": df.index.max(),
        })
    except Exception:
        pass

summary_df = pd.DataFrame(summary_rows)
print(f"Computed stats for {len(summary_df)} symbols out of {len(stat_sample)} sampled\n")
summary_df.describe().round(2)

In [ ]:
# Visualize distribution of average close prices and trading days
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(summary_df["mean_close"].clip(upper=500), bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Mean Close Price ($)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Average Close Prices (clipped at $500)")

axes[1].hist(summary_df["trading_days"], bins=50, edgecolor="black", alpha=0.7, color="green")
axes[1].set_xlabel("Number of Trading Days")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Data Length (trading days)")

axes[2].hist(np.log10(summary_df["avg_daily_volume"].clip(lower=1)), bins=50, edgecolor="black", alpha=0.7, color="orange")
axes[2].set_xlabel("log₁₀(Avg Daily Volume)")
axes[2].set_ylabel("Count")
axes[2].set_title("Distribution of Average Daily Volume (log scale)")

plt.tight_layout()
plt.show()

## 6. Filter Stocks by Date Range Coverage & Liquidity

Keep only symbols with **≥ 5 years of data** (≈1,260 trading days) and **average daily volume ≥ 100,000 shares**. This gives us a high-quality, liquid universe for the rest of the analysis.

In [ ]:
# To filter the full universe we need stats for ALL symbols (not just the 500 sample).
# This scans every parquet — may take a minute on a network drive.

print("Scanning all symbols for filtering (this may take a few minutes on Z: drive)...")
all_summary_rows = []
for i, sym in enumerate(all_symbols):
    try:
        df = pd.read_parquet(PARQUET_DIR / f"{sym}.parquet", columns=["close", "volume"])
        if len(df) == 0:
            continue
        all_summary_rows.append({
            "symbol": sym,
            "mean_close": df["close"].mean(),
            "avg_daily_volume": df["volume"].mean(),
            "trading_days": len(df),
            "start_date": df.index.min(),
            "end_date": df.index.max(),
        })
    except Exception:
        pass
    if (i + 1) % 2000 == 0:
        print(f"  ...scanned {i+1:,}/{len(all_symbols):,}")

all_summary_df = pd.DataFrame(all_summary_rows)
print(f"Scanned {len(all_summary_df):,} symbols successfully")

# Apply filters
MIN_TRADING_DAYS = 1260   # ~5 years
MIN_AVG_VOLUME = 100_000  # 100k shares/day

filtered = all_summary_df[
    (all_summary_df["trading_days"] >= MIN_TRADING_DAYS) &
    (all_summary_df["avg_daily_volume"] >= MIN_AVG_VOLUME)
].copy()

filtered_symbols = filtered["symbol"].tolist()

print(f"\n{'='*50}")
print(f"Universe before filter:  {len(all_summary_df):,}")
print(f"After ≥{MIN_TRADING_DAYS} days:       {(all_summary_df['trading_days'] >= MIN_TRADING_DAYS).sum():,}")
print(f"After ≥{MIN_AVG_VOLUME:,} avg vol:  {len(filtered):,}")
print(f"{'='*50}")
print(f"\nFiltered universe: {len(filtered_symbols)} symbols")
print(f"Sample: {filtered_symbols[:20]}")

## 7. Calculate Daily Returns for the Filtered Universe

In [ ]:
# Load close prices for the filtered universe and compute daily returns
# Use a cap to keep memory manageable — adjust as needed
MAX_SYMBOLS_FOR_RETURNS = 200
return_symbols = filtered_symbols[:MAX_SYMBOLS_FOR_RETURNS]

close_dict = {}
for sym in return_symbols:
    try:
        df = pd.read_parquet(PARQUET_DIR / f"{sym}.parquet", columns=["close"])
        close_dict[sym] = df["close"]
    except Exception:
        pass

close_wide = pd.DataFrame(close_dict)
returns_wide = close_wide.pct_change().dropna(how="all")

print(f"Returns matrix shape: {returns_wide.shape}  (dates × symbols)")
print(f"Date range: {returns_wide.index.min()} → {returns_wide.index.max()}\n")

# Summary return statistics across all symbols
return_stats = returns_wide.describe().T[["mean", "std", "min", "max"]]
return_stats.columns = ["Mean Daily Return", "Std (Volatility)", "Worst Day", "Best Day"]
return_stats.sort_values("Mean Daily Return", ascending=False).head(20)

## 8. Visualize Price Performance of Top N Stocks

Identify the top 10 stocks by total cumulative return and plot their normalized prices (rebased to 100) alongside their rolling 50-day volatility.

In [ ]:
# Find top 10 by cumulative return (using only symbols where we have start & end prices)
cum_returns = {}
for sym in close_wide.columns:
    prices = close_wide[sym].dropna()
    if len(prices) > 100:
        cum_returns[sym] = (prices.iloc[-1] / prices.iloc[0]) - 1

cum_ret_series = pd.Series(cum_returns).sort_values(ascending=False)
top10 = cum_ret_series.head(10).index.tolist()

print("Top 10 stocks by cumulative return:")
for rank, sym in enumerate(top10, 1):
    print(f"  {rank:2d}. {sym:6s}  {cum_ret_series[sym]:+.1%}")

# Plot normalized prices + rolling volatility
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for sym in top10:
    prices = close_wide[sym].dropna()
    normalized = prices / prices.iloc[0] * 100
    ax1.plot(normalized.index, normalized.values, label=sym, linewidth=1.2)

ax1.set_ylabel("Normalized Price (base = 100)")
ax1.set_title("Top 10 Stocks — Normalized Price Performance")
ax1.legend(ncol=5, fontsize=8)
ax1.grid(True, alpha=0.3)

for sym in top10:
    vol = returns_wide[sym].rolling(50).std() * np.sqrt(252) * 100
    ax2.plot(vol.index, vol.values, label=sym, linewidth=1, alpha=0.8)

ax2.set_ylabel("Annualized Volatility (%)")
ax2.set_xlabel("Date")
ax2.set_title("Top 10 Stocks — 50-Day Rolling Volatility")
ax2.legend(ncol=5, fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Correlation Matrix Across a Diverse Sample of Stocks

Select 20 liquid stocks from different price ranges. Compute pairwise return correlations and identify the most/least correlated pairs.

In [ ]:
# Pick 20 well-known, liquid symbols from our filtered set for a readable heatmap
# Sort filtered by avg volume and pick evenly across the list
filtered_sorted = filtered.sort_values("avg_daily_volume", ascending=False)
corr_step = max(1, len(filtered_sorted) // 20)
corr_symbols = filtered_sorted.iloc[::corr_step]["symbol"].tolist()[:20]

# Keep only those present in our returns matrix
corr_symbols = [s for s in corr_symbols if s in returns_wide.columns]
print(f"Correlation symbols ({len(corr_symbols)}): {corr_symbols}")

corr_matrix = returns_wide[corr_symbols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Daily Return Correlation Matrix — Diverse Stock Sample")
plt.tight_layout()
plt.show()

# Most and least correlated pairs
corr_pairs = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
corr_flat = corr_pairs.stack().sort_values()
print("\n🔗 Most correlated pairs:")
for (s1, s2), c in corr_flat.tail(5).items():
    print(f"  {s1} ↔ {s2}: {c:.3f}")
print("\n🔀 Least correlated pairs:")
for (s1, s2), c in corr_flat.head(5).items():
    print(f"  {s1} ↔ {s2}: {c:.3f}")

## 10. Universe-Wide Volume Trends Over Time

Aggregate total daily volume across all symbols in the filtered universe by month. Overlay the count of active symbols per month.

In [ ]:
# Load volume data for a broad sample (use the symbols already loaded in close_wide)
vol_dict = {}
for sym in return_symbols:
    try:
        df = pd.read_parquet(PARQUET_DIR / f"{sym}.parquet", columns=["volume"])
        vol_dict[sym] = df["volume"]
    except Exception:
        pass

volume_wide = pd.DataFrame(vol_dict)

# Monthly aggregation
monthly_total_volume = volume_wide.sum(axis=1).resample("MS").sum()
monthly_active_count = volume_wide.notna().sum(axis=1).resample("MS").mean()

fig, ax1 = plt.subplots(figsize=(14, 6))

color1 = "steelblue"
ax1.bar(monthly_total_volume.index, monthly_total_volume.values / 1e9,
        width=25, alpha=0.6, color=color1, label="Total Volume")
ax1.set_xlabel("Date")
ax1.set_ylabel("Monthly Total Volume (Billions of Shares)", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

ax2 = ax1.twinx()
color2 = "firebrick"
ax2.plot(monthly_active_count.index, monthly_active_count.values,
         color=color2, linewidth=2, label="Active Symbols")
ax2.set_ylabel("Avg Active Symbols per Month", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

ax1.set_title("Universe-Wide Volume Trends & Symbol Coverage Over Time")
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

## 11. Top Gainers & Losers Over a Configurable Period

Compute total return for every symbol in the filtered universe over a configurable lookback and display the top 20 gainers and losers.

In [ ]:
# --- CONFIG: Change these to adjust the lookback period ---
LOOKBACK_START = "2025-01-01"   # e.g. YTD
LOOKBACK_END   = None           # None = latest available date

# Compute returns for all symbols with data in this window
gainers_losers = []
for sym in filtered_symbols:
    try:
        df = pd.read_parquet(PARQUET_DIR / f"{sym}.parquet", columns=["close"])
        mask = df.index >= pd.Timestamp(LOOKBACK_START, tz="UTC")
        if LOOKBACK_END:
            mask &= df.index <= pd.Timestamp(LOOKBACK_END, tz="UTC")
        window = df.loc[mask]
        if len(window) < 10:
            continue
        start_price = window["close"].iloc[0]
        end_price = window["close"].iloc[-1]
        pct_change = (end_price / start_price - 1) * 100
        gainers_losers.append({
            "symbol": sym,
            "start_price": round(start_price, 2),
            "end_price": round(end_price, 2),
            "pct_change": round(pct_change, 2),
        })
    except Exception:
        pass

gl_df = pd.DataFrame(gainers_losers).sort_values("pct_change", ascending=False)
top20_gain = gl_df.head(20)
top20_lose = gl_df.tail(20).sort_values("pct_change")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

ax1.barh(top20_gain["symbol"], top20_gain["pct_change"], color="green", alpha=0.7)
ax1.set_xlabel("Return (%)")
ax1.set_title(f"Top 20 Gainers (since {LOOKBACK_START})")
ax1.invert_yaxis()

ax2.barh(top20_lose["symbol"], top20_lose["pct_change"], color="red", alpha=0.7)
ax2.set_xlabel("Return (%)")
ax2.set_title(f"Top 20 Losers (since {LOOKBACK_START})")
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

print(f"\n📈 Top 20 Gainers since {LOOKBACK_START}:")
print(top20_gain.to_string(index=False))
print(f"\n📉 Top 20 Losers since {LOOKBACK_START}:")
print(top20_lose.to_string(index=False))

## 12. Export Filtered Universe to Watchlist & Summary Files

Save the filtered symbol list and summary statistics for reuse in other notebooks and scripts.

In [ ]:
# Save filtered universe as a watchlist file
universe_dir = PROJECT_ROOT / "universes"
universe_dir.mkdir(exist_ok=True)

watchlist_path = universe_dir / "filtered_universe.txt"
with open(watchlist_path, "w") as f:
    f.write("\n".join(filtered_symbols))
print(f"✓ Saved {len(filtered_symbols)} symbols to {watchlist_path}")

# Save full summary stats to parquet for reuse
data_dir = PROJECT_ROOT / "data"
data_dir.mkdir(exist_ok=True)

summary_path = data_dir / "universe_summary_stats.parquet"
all_summary_df.to_parquet(summary_path, index=False)
print(f"✓ Saved summary stats to {summary_path}")

print(f"\nDone! Your EDA environment is ready for further exploration.")